# Pipeline final de análise espectral

## Parte 1 — importação, limpeza e pré-processamento

Esta etapa parte de `dataset/Unificada13052026.csv` e executa a limpeza de metadados e todo o pré-processamento espectral **neste notebook**. As próximas análises serão incorporadas em etapas posteriores.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.signal import savgol_filter


def localizar_raiz(inicio: Path | None = None) -> Path:
    """Permite iniciar o kernel na raiz do projeto ou em script_final."""
    inicio = (inicio or Path.cwd()).resolve()
    for candidato in (inicio, *inicio.parents):
        if (candidato / "dataset/Unificada13052026.csv").is_file():
            return candidato
    raise FileNotFoundError("Arquivo dataset/Unificada13052026.csv não encontrado.")


ROOT = localizar_raiz()
ARQUIVO_ENTRADA = ROOT / "dataset/Unificada13052026.csv"
print(f"Base de entrada: {ARQUIVO_ENTRADA}")


### 1. Limpeza dos metadados

A identificação registrada em `nomenclaura` corrige divergências em `genotipo` e `condicao`; o bloco passa a usar maiúsculas, e o erro `C202` é corrigido para `CD202`. Os valores espectrais permanecem intactos.


In [ ]:
def limpar_metadados(df: pd.DataFrame) -> pd.DataFrame:
    """Reproduz as quatro correções de dataset/limpar_metadados.py."""
    df = df.copy()
    df["bloco"] = df["bloco"].str.upper()

    tokens = df["nomenclaura"].str.split("_")
    typo_c202 = tokens.str[1] == "C202"
    tokens.loc[typo_c202] = tokens.loc[typo_c202].apply(
        lambda t: [t[0], "CD202", *t[2:]]
    )
    df.loc[typo_c202, "nomenclaura"] = tokens.loc[typo_c202].str.join("_")

    genotipo_nomenclaura = tokens.str[1]
    genotipos_validos = {"BR16", "CD202", "EMB48"}
    divergente = genotipo_nomenclaura.isin(genotipos_validos) & (
        genotipo_nomenclaura != df["genotipo"]
    )
    df.loc[divergente, "genotipo"] = genotipo_nomenclaura[divergente]

    condicoes_validas = {
        "IRRIG": "IRRIG", "NIRRIG": "NIRRIG",
        "IRR": "IRRIG", "NIRR": "NIRRIG",
    }
    condicao_nomenclaura = tokens.str[2].map(condicoes_validas)
    divergente = condicao_nomenclaura.notna() & (
        condicao_nomenclaura != df["condicao"]
    )
    df.loc[divergente, "condicao"] = condicao_nomenclaura[divergente]
    return df


In [ ]:
base_bruta = pd.read_csv(ARQUIVO_ENTRADA, sep=";", decimal=",")
base_bruta.columns = base_bruta.columns.str.strip()
base_limpa = limpar_metadados(base_bruta)

colunas_metadados = [c for c in base_limpa.columns if not c.isdigit()]
colunas_espectrais = sorted((c for c in base_limpa.columns if c.isdigit()), key=int)
correcoes = {
    coluna: int((base_bruta[coluna] != base_limpa[coluna]).sum())
    for coluna in ("bloco", "genotipo", "nomenclaura", "condicao")
}
print(f"Amostras: {len(base_limpa)} | Metadados: {len(colunas_metadados)}")
print(f"Bandas originais: {len(colunas_espectrais)} ({colunas_espectrais[0]}–{colunas_espectrais[-1]} nm)")
display(pd.DataFrame(correcoes.items(), columns=["metadado", "linhas corrigidas"]))


### 2. Pré-processamento espectral

O fluxo usa as mesmas regras do projeto: recorte de 400 a 2450 nm, limite absoluto de 10.000 para valores extremos, correção das emendas dos detectores em 1000 e 1800 nm, interpolação dos ausentes, filtro Savitzky–Golay (janela 11, polinômio 2) e SNV por amostra.


In [ ]:
LIMITE_INF, LIMITE_SUP = 400, 2450
SPLICE_POINTS = (1000, 1800)
REFERENCE_SEQ = 1
CLIP_ABS = 1e4
WINDOW_LENGTH, POLYORDER, MODE = 11, 2, "interp"


def jump_correct(series: pd.Series) -> pd.Series:
    """Alinha os três detectores por deslocamento nas emendas."""
    series = series.sort_index()

    def get_seq(wl: float) -> int:
        for i, splice in enumerate(SPLICE_POINTS):
            if wl <= splice:
                return i
        return len(SPLICE_POINTS)

    groups = series.groupby(get_seq)

    def shift(ref: pd.Series, mov: pd.Series, right: bool = True) -> pd.Series:
        diff = (ref.iloc[-1] - mov.iloc[0]) if right else (ref.iloc[0] - mov.iloc[-1])
        return mov + diff

    for i in range(REFERENCE_SEQ, groups.ngroups - 1):
        series.update(shift(groups.get_group(i), groups.get_group(i + 1)))
    for i in range(REFERENCE_SEQ, 0, -1):
        series.update(shift(groups.get_group(i), groups.get_group(i - 1), right=False))
    return series


def limpar_valores_espurios(espectro: np.ndarray) -> np.ndarray:
    espectro = np.clip(espectro.copy(), -CLIP_ABS, CLIP_ABS)
    espectro[~np.isfinite(espectro)] = np.nan
    return espectro


def interpolar(espectro: np.ndarray) -> np.ndarray:
    espectro = espectro.copy()
    for i in range(espectro.shape[0]):
        linha = espectro[i]
        if np.isnan(linha).any():
            espectro[i] = pd.Series(linha).interpolate(limit_direction="both").to_numpy()
    return np.nan_to_num(espectro, nan=0.0, posinf=0.0, neginf=0.0)


def preprocessar(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """Recorta, limpa, corrige as emendas e interpola os espectros."""
    meta = [c for c in df.columns if not c.isdigit()]
    bandas = sorted(int(c) for c in df.columns if c.isdigit())
    bandas_validas = [b for b in bandas if LIMITE_INF <= b <= LIMITE_SUP]
    if not bandas_validas:
        raise ValueError("Nenhuma banda entre 400 e 2450 nm foi encontrada.")
    w = np.array(bandas_validas, dtype=float)

    espectro = df[[str(b) for b in bandas_validas]].to_numpy(dtype=float)
    espectro_limpo = limpar_valores_espurios(espectro)
    espectro_corrigido = np.zeros_like(espectro_limpo)
    for i in range(espectro_limpo.shape[0]):
        serie = pd.Series(espectro_limpo[i], index=w, dtype="float64")
        espectro_corrigido[i] = jump_correct(serie).to_numpy()

    df_espectro = pd.DataFrame(
        interpolar(espectro_corrigido),
        columns=[str(b) for b in bandas_validas], index=df.index,
    )
    return pd.concat([df[meta], df_espectro], axis=1), w


def savitzky_golay(espectro: np.ndarray) -> np.ndarray:
    """Suaviza cada amostra com Savitzky–Golay."""
    window = WINDOW_LENGTH + (WINDOW_LENGTH % 2 == 0)
    min_window = POLYORDER + 2 if (POLYORDER + 2) % 2 else POLYORDER + 3
    window = max(window, min_window)
    suavizado = np.zeros_like(espectro)
    for i, linha in enumerate(espectro):
        suavizado[i] = (
            savgol_filter(linha, window, POLYORDER, mode=MODE)
            if len(linha) >= window else linha
        )
    return suavizado


def snv(espectro: np.ndarray) -> np.ndarray:
    """Centraliza e escala cada espectro por seu desvio padrão."""
    normalizado = np.zeros_like(espectro)
    for i, linha in enumerate(espectro):
        media, desvio = np.nanmean(linha), np.nanstd(linha)
        normalizado[i] = (
            (linha - media) / desvio
            if desvio > 0 and np.isfinite(desvio) else linha - media
        )
    return normalizado


In [ ]:
base_recortada, comprimentos_onda = preprocessar(base_limpa)
bandas_processadas = [str(int(w)) for w in comprimentos_onda]

base_suavizada = base_recortada.copy()
base_suavizada[bandas_processadas] = savitzky_golay(
    base_recortada[bandas_processadas].to_numpy(dtype=float)
)

base_normalizada = base_suavizada.copy()
base_normalizada[bandas_processadas] = snv(
    base_suavizada[bandas_processadas].to_numpy(dtype=float)
)

print(f"Pré-processamento concluído: {len(base_normalizada)} amostras, "
      f"{len(bandas_processadas)} bandas ({int(comprimentos_onda[0])}–{int(comprimentos_onda[-1])} nm).")


In [ ]:
resumo_preprocessamento = pd.DataFrame([
    {
        "estágio": nome,
        "amostras": len(base),
        "bandas": len(bandas_processadas),
        "ausentes": int(base[bandas_processadas].isna().sum().sum()),
    }
    for nome, base in (
        ("corrigido e recortado", base_recortada),
        ("suavizado", base_suavizada),
        ("normalizado (SNV)", base_normalizada),
    )
])
display(resumo_preprocessamento)


## Parte 2 — dispersão espectral por genótipo e dia

Uma figura por genótipo compara os sete dias de coleta matinal. Cada linha mostra um dia; as colunas mostram o espectro corrigido e recortado e o espectro suavizado e normalizado. Azul representa IRRIG e laranja, NIRRIG. A linha contínua é a média, as tracejadas delimitam o intervalo de confiança de 95% da média, e as pontilhadas mostram o mínimo e o máximo das leituras em cada banda.

Os gráficos usam diretamente `base_recortada` e `base_normalizada`, geradas acima. Nenhum script ou CSV intermediário é carregado.


In [ ]:
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import t as t_dist

CONDICOES = ("IRRIG", "NIRRIG")
COR_CONDICAO = {"IRRIG": "#2a78d6", "NIRRIG": "#eb6834"}
CONFIANCA = 0.95
ESTAGIOS_GRAFICO = (
    ("recortado", "Recortado (recorte + correção das emendas)", "Reflectância"),
    ("normalizado", "Normalizado (Savitzky–Golay + SNV)", "Reflectância (SNV)"),
)
EXIBIR_GRAFICOS = True
SALVAR_GRAFICOS = True
PASTA_GRAFICOS = ROOT / "script_final/outputs/dispersoes"

mascara_manha = base_limpa["turno"].eq("manha")
meta_graficos = base_limpa.loc[
    mascara_manha, ["genotipo", "condicao", "data_coleta", "turno"]
].copy().reset_index(drop=True)
meta_graficos["dia"] = meta_graficos["data_coleta"].astype(str).str.extract(r"^(D\d+)")
if meta_graficos["dia"].isna().any():
    raise ValueError("Há códigos de data_coleta sem dia no formato Dxx.")

espectros_graficos = {
    "recortado": base_recortada.loc[mascara_manha, bandas_processadas].to_numpy(dtype=float),
    "normalizado": base_normalizada.loc[mascara_manha, bandas_processadas].to_numpy(dtype=float),
}
dias_graficos = sorted(meta_graficos["dia"].unique())
genotipos_graficos = sorted(meta_graficos["genotipo"].unique())
if not dias_graficos or not genotipos_graficos:
    raise ValueError("Nenhuma amostra matinal encontrada para os gráficos.")

contagens_graficos = (
    meta_graficos.groupby(["genotipo", "dia", "condicao"]).size()
    .rename("leituras").reset_index()
)
display(contagens_graficos.pivot(index=["genotipo", "dia"], columns="condicao", values="leituras"))


In [ ]:
def resumo_banda(espectro: np.ndarray) -> dict[str, np.ndarray | int]:
    """Média, IC t de Student de 95% da média e extremos por banda."""
    n = espectro.shape[0]
    if n == 0:
        raise ValueError("Não é possível resumir um grupo sem leituras.")
    media = espectro.mean(axis=0)
    minimo = espectro.min(axis=0)
    maximo = espectro.max(axis=0)
    if n > 1:
        erro = espectro.std(axis=0, ddof=1) / np.sqrt(n)
        margem = t_dist.ppf(0.5 + CONFIANCA / 2, df=n - 1) * erro
    else:
        margem = np.zeros_like(media)
    return {
        "media": media, "ic_inf": media - margem, "ic_sup": media + margem,
        "min": minimo, "max": maximo, "n": n,
    }


def plot_celula(ax: plt.Axes, w: np.ndarray, resumos: dict, titulo: str, unidade: str) -> None:
    """Desenha as duas condições em um dia e estágio."""
    for condicao in CONDICOES:
        if condicao not in resumos:
            continue
        cor, r = COR_CONDICAO[condicao], resumos[condicao]
        ax.plot(w, r["min"], color=cor, linewidth=0.7, linestyle=(0, (1, 2)))
        ax.plot(w, r["max"], color=cor, linewidth=0.7, linestyle=(0, (1, 2)))
        ax.plot(w, r["ic_inf"], color=cor, linewidth=1.1, linestyle=(0, (6, 4)))
        ax.plot(w, r["ic_sup"], color=cor, linewidth=1.1, linestyle=(0, (6, 4)))

    for condicao in CONDICOES:
        if condicao not in resumos:
            continue
        ax.plot(
            w, resumos[condicao]["media"], color=COR_CONDICAO[condicao],
            linewidth=2.2, solid_capstyle="round",
            path_effects=[pe.Stroke(linewidth=3.0, foreground="white"), pe.Normal()],
        )
    ax.set_title(titulo, fontsize=10, fontweight="bold")
    ax.set_xlabel("Comprimento de onda (nm)", fontsize=8)
    ax.set_ylabel(unidade, fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=7)


def legenda_figura(fig: plt.Figure) -> None:
    handles = [
        Line2D([], [], color=COR_CONDICAO[c], linewidth=2.2, label=c)
        for c in CONDICOES
    ] + [
        Line2D([], [], color="#555555", linewidth=2.2, label="Média"),
        Line2D([], [], color="#555555", linewidth=1.1, linestyle=(0, (6, 4)),
               label="IC 95% da média"),
        Line2D([], [], color="#555555", linewidth=0.7, linestyle=(0, (1, 2)),
               label="Mínimo / máximo"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles),
               frameon=False, fontsize=10)


def gerar_figura_genotipo(genotipo: str) -> plt.Figure:
    """Cria o painel 7 × 2 de um genótipo usando os espectros já processados."""
    do_genotipo = meta_graficos["genotipo"].eq(genotipo).to_numpy()
    resumos = {}
    for chave, _, _ in ESTAGIOS_GRAFICO:
        for dia in dias_graficos:
            do_dia = do_genotipo & meta_graficos["dia"].eq(dia).to_numpy()
            por_condicao = {}
            for condicao in CONDICOES:
                mascara = do_dia & meta_graficos["condicao"].eq(condicao).to_numpy()
                if mascara.any():
                    por_condicao[condicao] = resumo_banda(espectros_graficos[chave][mascara])
            if por_condicao:
                resumos[(chave, dia)] = por_condicao

    limites = {}
    for chave, _, _ in ESTAGIOS_GRAFICO:
        valores = np.concatenate([
            r[k] for (estagio, _), por_condicao in resumos.items() if estagio == chave
            for r in por_condicao.values() for k in ("min", "max")
        ])
        folga = 0.05 * (valores.max() - valores.min())
        limites[chave] = (valores.min() - folga, valores.max() + folga)

    with plt.style.context("seaborn-v0_8-whitegrid"):
        fig, axes = plt.subplots(
            len(dias_graficos), len(ESTAGIOS_GRAFICO),
            figsize=(7.5 * len(ESTAGIOS_GRAFICO), 3.4 * len(dias_graficos)),
            squeeze=False,
        )
        for i, dia in enumerate(dias_graficos):
            for j, (chave, rotulo, unidade) in enumerate(ESTAGIOS_GRAFICO):
                ax = axes[i, j]
                resumo = resumos.get((chave, dia))
                if resumo is None:
                    ax.set_axis_off()
                    continue
                ns = ", ".join(
                    f"{c} n={resumo[c]['n']}" for c in CONDICOES if c in resumo
                )
                plot_celula(ax, comprimentos_onda, resumo,
                            f"{dia} - {rotulo} ({ns})", unidade)
                ax.set_ylim(*limites[chave])
        legenda_figura(fig)
        fig.suptitle(
            f"Genótipo {genotipo} — dispersão espectral dia a dia\n"
            "Turno da manhã; escala vertical compartilhada por estágio",
            fontsize=14, fontweight="bold",
        )
        fig.tight_layout(rect=[0, 0.025, 1, 0.975])
    return fig


In [ ]:
if SALVAR_GRAFICOS:
    PASTA_GRAFICOS.mkdir(parents=True, exist_ok=True)

arquivos_graficos = {}
for genotipo in genotipos_graficos:
    figura = gerar_figura_genotipo(genotipo)
    if SALVAR_GRAFICOS:
        arquivo = PASTA_GRAFICOS / f"dispersao_dia_{genotipo}.png"
        figura.savefig(arquivo, dpi=150, bbox_inches="tight")
        arquivos_graficos[genotipo] = arquivo
        print(f"{genotipo}: {arquivo.relative_to(ROOT)}")
    if EXIBIR_GRAFICOS:
        display(figura)
    plt.close(figura)


## Parte 3 — classificação IRRIG × NIRRIG por dia

Os três genótipos são reunidos **dentro de cada dia**; somente as leituras da manhã entram na análise. Em cada dia, bandas contíguas com correlação de Spearman `|ρ| > 0,80` são agrupadas por ligação completa dentro de uma janela menor que 10 nm. A banda central de cada grupo representa a região. Uma Random Forest ajustada naquele dia ranqueia essas representantes por importância Gini e define as cinco bandas do modelo XGBoost daquele dia. **NIRRIG é a classe positiva.**

Para avaliar a classificação, cada um dos quatro blocos fica fora do treino uma vez. A redução por Spearman e a seleção por Random Forest são refeitas **somente com os dados de treino de cada dobra**; o XGBoost prevê as leituras do bloco excluído. Assim, as métricas de validação não usam bandas escolhidas com acesso ao bloco de teste. A tabela de importância exibida para cada dia vem de um ajuste final com todas as leituras daquele dia e determina as cinco bandas do modelo final correspondente.


In [ ]:
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from xgboost import XGBClassifier

LIMIAR_SPEARMAN = 0.80
JANELA_SPEARMAN_NM = 10.0
TOP_K = 5
N_ARVORES_RF = 200
N_ARVORES_XGB = 150
SEMENTE_CLASSIFICACAO = 42
PASTA_CLASSIFICACAO = ROOT / "script_final/outputs/classificacao"


def reduzir_spearman(X: np.ndarray, w: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Agrupa bandas vizinhas com |ρ| > 0,80 contra todos os membros do grupo."""
    postos = rankdata(X, axis=0, method="average")
    z = postos - postos.mean(axis=0)
    norma = np.sqrt((z ** 2).sum(axis=0))
    norma[norma == 0] = np.inf
    z /= norma

    grupos = np.zeros(len(w), dtype=int)
    membros = [0]
    atual = 0
    for j in range(1, len(w)):
        dentro_janela = w[j] - w[membros[0]] < JANELA_SPEARMAN_NM
        correlacionada = dentro_janela and np.all(
            np.abs(z[:, membros].T @ z[:, j]) > LIMIAR_SPEARMAN
        )
        if correlacionada:
            membros.append(j)
        else:
            atual += 1
            membros = [j]
        grupos[j] = atual

    representantes = np.array([
        indices[len(indices) // 2]
        for grupo in range(atual + 1)
        if (indices := np.flatnonzero(grupos == grupo)).size
    ], dtype=int)
    return grupos, representantes


def selecionar_top5_rf(
    X: np.ndarray, y: np.ndarray, w: np.ndarray, semente: int
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    """Reduz bandas e ranqueia as representantes pela importância Gini da RF."""
    grupos, representantes = reduzir_spearman(X, w)
    if len(representantes) < TOP_K:
        raise ValueError(f"A redução de Spearman deixou apenas {len(representantes)} bandas.")
    rf = RandomForestClassifier(
        n_estimators=N_ARVORES_RF, max_features="sqrt",
        class_weight="balanced", random_state=semente, n_jobs=1,
    ).fit(X[:, representantes], y)
    importancia = rf.feature_importances_
    ordem = np.lexsort((w[representantes], -importancia))
    tabela = pd.DataFrame({
        "posicao": np.arange(1, len(ordem) + 1),
        "banda_nm": w[representantes][ordem].astype(int),
        "importancia_gini": importancia[ordem],
    })
    return tabela, representantes[ordem[:TOP_K]], grupos, representantes


def novo_xgboost(y_treino: np.ndarray, semente: int) -> XGBClassifier:
    negativos = int((y_treino == 0).sum())
    positivos = int((y_treino == 1).sum())
    return XGBClassifier(
        n_estimators=N_ARVORES_XGB, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=1.0, reg_lambda=1.0,
        scale_pos_weight=negativos / positivos,
        objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", n_jobs=1, random_state=semente,
    )


def calcular_metricas(y_real: np.ndarray, y_pred: np.ndarray, prob: np.ndarray) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return {
        "acuracia": accuracy_score(y_real, y_pred),
        "acuracia_balanceada": balanced_accuracy_score(y_real, y_pred),
        "precisao_nirrig": precision_score(y_real, y_pred, zero_division=0),
        "sensibilidade_nirrig": recall_score(y_real, y_pred, zero_division=0),
        "f1_nirrig": f1_score(y_real, y_pred, zero_division=0),
        "auc_roc": roc_auc_score(y_real, prob),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


In [ ]:
# Este ramo usa somente as bases produzidas na parte 1, sem arquivos intermediários.
mascara_classificacao = base_limpa["turno"].eq("manha")
meta_classificacao = base_limpa.loc[
    mascara_classificacao, ["bloco", "genotipo", "condicao", "data_coleta"]
].copy().reset_index(drop=True)
meta_classificacao["dia"] = (
    meta_classificacao["data_coleta"].astype(str).str.extract(r"^(D\d+)")
)
if meta_classificacao["dia"].isna().any():
    raise ValueError("Há códigos de data_coleta sem dia no formato Dxx.")
X_classificacao = base_normalizada.loc[
    mascara_classificacao, bandas_processadas
].to_numpy(dtype=float)
y_classificacao = meta_classificacao["condicao"].eq("NIRRIG").astype(int).to_numpy()
if not set(meta_classificacao["condicao"].unique()).issubset({"IRRIG", "NIRRIG"}):
    raise ValueError("Foram encontradas condições diferentes de IRRIG e NIRRIG.")

PASTA_CLASSIFICACAO.mkdir(parents=True, exist_ok=True)
importancias_rf_por_dia = {}
modelos_xgboost_por_dia = {}
top5_tabelas, metricas_dias, metricas_dobras, predicoes_dias = [], [], [], []

for numero_dia, dia in enumerate(sorted(meta_classificacao["dia"].unique())):
    mascara_dia = meta_classificacao["dia"].eq(dia).to_numpy()
    X_dia = X_classificacao[mascara_dia]
    y_dia = y_classificacao[mascara_dia]
    meta_dia = meta_classificacao.loc[mascara_dia].reset_index(drop=True)
    blocos = meta_dia["bloco"].to_numpy()
    if len(np.unique(blocos)) != 4 or set(np.unique(y_dia)) != {0, 1}:
        raise ValueError(f"{dia}: esperados quatro blocos e as duas condições.")

    # Importância descritiva e modelo final: ajuste independente em cada dia.
    importancia, top_indices, grupos, representantes = selecionar_top5_rf(
        X_dia, y_dia, comprimentos_onda,
        SEMENTE_CLASSIFICACAO + numero_dia,
    )
    importancias_rf_por_dia[dia] = importancia
    importancia.to_csv(PASTA_CLASSIFICACAO / f"importancia_rf_{dia}.csv", sep=";", index=False)
    pd.DataFrame({
        "banda_nm": comprimentos_onda.astype(int),
        "grupo_spearman": grupos,
        "representante_nm": comprimentos_onda[representantes][grupos].astype(int),
    }).to_csv(PASTA_CLASSIFICACAO / f"grupos_spearman_{dia}.csv", sep=";", index=False)
    top = importancia.head(TOP_K).copy()
    top.insert(0, "dia", dia)
    top5_tabelas.append(top)
    modelo_final = novo_xgboost(y_dia, SEMENTE_CLASSIFICACAO + numero_dia)
    modelo_final.fit(X_dia[:, top_indices], y_dia)
    modelos_xgboost_por_dia[dia] = modelo_final

    # Validação por bloco: refaz Spearman e RF apenas nas observações de treino.
    previsto = np.full(len(y_dia), -1, dtype=int)
    probabilidade = np.full(len(y_dia), np.nan)
    dobra_por_linha = np.full(len(y_dia), -1, dtype=int)
    for dobra, (treino, teste) in enumerate(
        GroupKFold(n_splits=4).split(X_dia, y_dia, groups=blocos), start=1
    ):
        assert not set(blocos[treino]) & set(blocos[teste])
        _, top_treino, _, _ = selecionar_top5_rf(
            X_dia[treino], y_dia[treino], comprimentos_onda,
            SEMENTE_CLASSIFICACAO + 100 * numero_dia + dobra,
        )
        modelo = novo_xgboost(y_dia[treino], SEMENTE_CLASSIFICACAO + 100 * numero_dia + dobra)
        modelo.fit(X_dia[np.ix_(treino, top_treino)], y_dia[treino])
        previsto[teste] = modelo.predict(X_dia[np.ix_(teste, top_treino)]).astype(int)
        probabilidade[teste] = modelo.predict_proba(X_dia[np.ix_(teste, top_treino)])[:, 1]
        dobra_por_linha[teste] = dobra
        metricas_dobras.append({
            "dia": dia, "dobra": dobra, "bloco_teste": ", ".join(sorted(set(blocos[teste]))),
            "n_treino": len(treino), "n_teste": len(teste),
            "bandas_treino_nm": ", ".join(map(str, comprimentos_onda[top_treino].astype(int))),
            **calcular_metricas(y_dia[teste], previsto[teste], probabilidade[teste]),
        })

    assert (previsto >= 0).all() and np.isfinite(probabilidade).all()
    metricas_dias.append({
        "dia": dia, "amostras": len(y_dia), "genotipos": meta_dia["genotipo"].nunique(),
        "bandas_representantes": len(representantes),
        "top5_modelo_final_nm": ", ".join(map(str, top["banda_nm"])),
        **calcular_metricas(y_dia, previsto, probabilidade),
    })
    predicoes_dias.append(pd.DataFrame({
        "dia": dia, "bloco": blocos, "genotipo": meta_dia["genotipo"],
        "condicao_real": meta_dia["condicao"], "classe_real_nirrig": y_dia,
        "classe_predita_nirrig": previsto, "probabilidade_nirrig": probabilidade,
        "dobra": dobra_por_linha,
    }))
    print(f"{dia}: {len(y_dia)} leituras, {len(representantes)} representantes; "
          f"Top 5: {', '.join(map(str, top['banda_nm']))}")

top5_rf_por_dia = pd.concat(top5_tabelas, ignore_index=True)
metricas_xgboost_por_dia = pd.DataFrame(metricas_dias)
metricas_xgboost_por_dobra = pd.DataFrame(metricas_dobras)
predicoes_xgboost = pd.concat(predicoes_dias, ignore_index=True)

top5_rf_por_dia.to_csv(PASTA_CLASSIFICACAO / "top5_rf_por_dia.csv", sep=";", index=False)
metricas_xgboost_por_dia.to_csv(PASTA_CLASSIFICACAO / "metricas_xgboost_por_dia.csv", sep=";", index=False)
metricas_xgboost_por_dobra.to_csv(PASTA_CLASSIFICACAO / "metricas_xgboost_por_dobra.csv", sep=";", index=False)
predicoes_xgboost.to_csv(PASTA_CLASSIFICACAO / "predicoes_xgboost.csv", sep=";", index=False)


In [ ]:
display(top5_rf_por_dia.round({"importancia_gini": 4}))
colunas_metricas = [
    "dia", "amostras", "bandas_representantes", "acuracia",
    "acuracia_balanceada", "f1_nirrig", "auc_roc",
]
display(metricas_xgboost_por_dia[colunas_metricas].round(3))

fig, eixos = plt.subplots(len(importancias_rf_por_dia), 1, figsize=(10, 2.7 * len(importancias_rf_por_dia)))
for eixo, (dia, importancia) in zip(eixos, importancias_rf_por_dia.items()):
    top = importancia.head(TOP_K).iloc[::-1]
    eixo.barh(top["banda_nm"].astype(str), top["importancia_gini"], color="#2a78d6")
    eixo.set_title(f"{dia} — importância Random Forest", loc="left", fontweight="bold")
    eixo.set_xlabel("Importância Gini")
    eixo.grid(axis="x", alpha=0.25)
fig.tight_layout(h_pad=2.0)
arquivo_importancias = PASTA_CLASSIFICACAO / "importancia_top5_por_dia.png"
fig.savefig(arquivo_importancias, dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)
print(f"Resultados salvos em: {PASTA_CLASSIFICACAO.relative_to(ROOT)}")


### 3.1. Matrizes de confusão e estatísticas por dia

As matrizes abaixo usam **somente as previsões dos blocos deixados fora do treino** na validação da parte 3. Linhas são as condições reais e colunas são as condições previstas. Cada célula mostra a contagem de leituras e a porcentagem dentro da condição real. As estatísticas por classe incluem precisão, sensibilidade (recall), F1 e suporte; NIRRIG continua sendo a classe positiva para AUC-ROC e para as métricas binárias da tabela geral.


In [ ]:
from sklearn.metrics import classification_report

classes = ("IRRIG", "NIRRIG")
matrizes_confusao_por_dia = {}
linhas_matrizes, linhas_estatisticas = [], []

for dia in sorted(predicoes_xgboost["dia"].unique()):
    dados_dia = predicoes_xgboost.loc[predicoes_xgboost["dia"].eq(dia)]
    real = dados_dia["classe_real_nirrig"].to_numpy(dtype=int)
    previsto = dados_dia["classe_predita_nirrig"].to_numpy(dtype=int)
    matriz = confusion_matrix(real, previsto, labels=[0, 1])
    matrizes_confusao_por_dia[dia] = matriz

    linha_metricas = metricas_xgboost_por_dia.set_index("dia").loc[dia]
    assert matriz.tolist() == [
        [int(linha_metricas["tn"]), int(linha_metricas["fp"])],
        [int(linha_metricas["fn"]), int(linha_metricas["tp"])],
    ], f"Matriz de {dia} divergente das métricas da validação."
    assert matriz.sum() == len(dados_dia)

    matriz_dia = pd.DataFrame(matriz, index=classes, columns=classes)
    matriz_dia.index.name = "condicao_real"
    matriz_dia.columns.name = "condicao_predita"
    matriz_dia.to_csv(PASTA_CLASSIFICACAO / f"matriz_confusao_{dia}.csv", sep=";")
    for i, condicao_real in enumerate(classes):
        linhas_matrizes.append({
            "dia": dia, "condicao_real": condicao_real,
            "previsto_IRRIG": int(matriz[i, 0]),
            "previsto_NIRRIG": int(matriz[i, 1]),
        })

    relatorio = classification_report(
        real, previsto, labels=[0, 1], target_names=list(classes),
        output_dict=True, zero_division=0,
    )
    for classe in (*classes, "macro avg", "weighted avg"):
        valores = relatorio[classe]
        linhas_estatisticas.append({
            "dia": dia, "classe": classe,
            "precisao": valores["precision"],
            "sensibilidade_recall": valores["recall"],
            "f1": valores["f1-score"],
            "suporte": int(valores["support"]),
        })

matrizes_confusao_tabela = pd.DataFrame(linhas_matrizes)
estatisticas_por_classe_dia = pd.DataFrame(linhas_estatisticas)
matrizes_confusao_tabela.to_csv(
    PASTA_CLASSIFICACAO / "matrizes_confusao_por_dia.csv", sep=";", index=False
)
estatisticas_por_classe_dia.to_csv(
    PASTA_CLASSIFICACAO / "estatisticas_por_classe_dia.csv", sep=";", index=False
)

display(metricas_xgboost_por_dia[[
    "dia", "amostras", "acuracia", "acuracia_balanceada",
    "precisao_nirrig", "sensibilidade_nirrig", "f1_nirrig", "auc_roc",
]].round(3))
display(estatisticas_por_classe_dia.round(3))


In [ ]:
dias_matrizes = sorted(matrizes_confusao_por_dia)
fig, eixos = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
maximo = max(int(matriz.max()) for matriz in matrizes_confusao_por_dia.values())

for eixo, dia in zip(eixos.flat, dias_matrizes):
    matriz = matrizes_confusao_por_dia[dia]
    eixo.imshow(matriz, cmap="Blues", vmin=0, vmax=maximo)
    eixo.set_title(f"{dia}  ·  n={matriz.sum()}", fontweight="bold")
    eixo.set_xticks([0, 1], classes)
    eixo.set_yticks([0, 1], classes)
    eixo.set_xlabel("Predita")
    eixo.set_ylabel("Real")
    for linha in range(2):
        total_real = matriz[linha].sum()
        for coluna in range(2):
            n = int(matriz[linha, coluna])
            percentual = 100 * n / total_real if total_real else 0
            eixo.text(
                coluna, linha, f"{n}\n({percentual:.1f}%)",
                ha="center", va="center", fontsize=11,
                color="white" if n > maximo / 2 else "#1d2a35",
            )

for eixo in list(eixos.flat)[len(dias_matrizes):]:
    eixo.axis("off")
arquivo_matrizes = PASTA_CLASSIFICACAO / "matrizes_confusao_por_dia.png"
fig.savefig(arquivo_matrizes, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)

fig, eixo = plt.subplots(figsize=(10, 5))
for coluna, rotulo, cor in (
    ("acuracia", "Acurácia", "#2a78d6"),
    ("acuracia_balanceada", "Acurácia balanceada", "#1baf7a"),
    ("f1_nirrig", "F1 NIRRIG", "#eb6834"),
    ("auc_roc", "AUC-ROC", "#7554a3"),
):
    eixo.plot(
        metricas_xgboost_por_dia["dia"], metricas_xgboost_por_dia[coluna],
        marker="o", linewidth=2, label=rotulo, color=cor,
    )
eixo.set_ylim(0, 1.03)
eixo.set_xlabel("Dia")
eixo.set_ylabel("Valor da métrica")
eixo.set_title("Classificação por dia — validação por bloco", fontweight="bold")
eixo.grid(alpha=0.25)
eixo.legend(ncol=2, frameon=False)
fig.tight_layout()
arquivo_metricas = PASTA_CLASSIFICACAO / "metricas_xgboost_por_dia.png"
fig.savefig(arquivo_metricas, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)


### 3.2. Otimização do XGBoost com Optuna

A seleção Spearman → Random Forest → Top 5 continua independente para cada dia. O Optuna ajusta apenas os hiperparâmetros do XGBoost, maximizando a **acurácia balanceada** em validação interna por bloco. Em cada divisão interna, Spearman e Random Forest são calculados novamente apenas no treino interno.

A comparação com o XGBoost anterior usa os mesmos quatro blocos externos deixados de fora. Para cada bloco externo, a busca do Optuna vê somente os outros três blocos; sua previsão do bloco de teste é reunida às demais previsões fora da amostra. Depois da avaliação, uma busca separada com os quatro blocos escolhe os parâmetros do modelo final de cada dia, ajustado às Top 5 bandas daquele dia. A otimização pode melhorar alguns dias e piorar outros; a tabela abaixo mostra ambos.


In [ ]:
import optuna

N_TRIALS_OPTUNA = 20
optuna.logging.set_verbosity(optuna.logging.WARNING)

PARAMETROS_BASELINE = {
    "n_estimators": 150,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 1.0,
    "min_child_weight": 1.0,
    "reg_lambda": 1.0,
}


def preparar_validacao_interna(
    X: np.ndarray, y: np.ndarray, blocos: np.ndarray,
    w: np.ndarray, semente: int,
) -> list[tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """Seleciona bandas dentro de cada treino interno, uma vez por dobra."""
    partes = []
    n_blocos = len(np.unique(blocos))
    for dobra, (treino, validacao) in enumerate(
        GroupKFold(n_splits=n_blocos).split(X, y, groups=blocos), start=1
    ):
        assert not set(blocos[treino]) & set(blocos[validacao])
        _, bandas, _, _ = selecionar_top5_rf(
            X[treino], y[treino], w, semente + dobra,
        )
        partes.append((
            X[np.ix_(treino, bandas)], y[treino],
            X[np.ix_(validacao, bandas)], y[validacao],
        ))
    return partes


def otimizar_parametros_xgboost(partes: list, semente: int) -> optuna.Study:
    """Busca reprodutível por parâmetros, com seleção de bandas já isolada nas dobras."""
    def objetivo(trial: optuna.Trial) -> float:
        parametros = {
            "n_estimators": trial.suggest_int("n_estimators", 80, 350),
            "max_depth": trial.suggest_int("max_depth", 2, 5),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.20, log=True),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
        }
        pontuacoes = []
        for dobra, (X_treino, y_treino, X_validacao, y_validacao) in enumerate(partes):
            modelo = novo_xgboost(y_treino, semente + dobra).set_params(**parametros)
            modelo.fit(X_treino, y_treino)
            previsto = modelo.predict(X_validacao).astype(int)
            pontuacoes.append(balanced_accuracy_score(y_validacao, previsto))
        return float(np.mean(pontuacoes))

    estudo = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=semente),
    )
    estudo.enqueue_trial(PARAMETROS_BASELINE)
    estudo.optimize(objetivo, n_trials=N_TRIALS_OPTUNA, n_jobs=1, show_progress_bar=False)
    return estudo


In [ ]:
pasta_optuna = PASTA_CLASSIFICACAO / "optuna"
pasta_optuna.mkdir(parents=True, exist_ok=True)

linhas_parametros_dobras = []
linhas_parametros_finais = []
linhas_metricas_optuna = []
predicoes_optuna_lista = []
modelos_xgboost_optuna_por_dia = {}

for numero_dia, dia in enumerate(sorted(meta_classificacao["dia"].unique())):
    mascara_dia = meta_classificacao["dia"].eq(dia).to_numpy()
    X_dia = X_classificacao[mascara_dia]
    y_dia = y_classificacao[mascara_dia]
    meta_dia = meta_classificacao.loc[mascara_dia].reset_index(drop=True)
    blocos = meta_dia["bloco"].to_numpy()
    referencia = predicoes_xgboost.loc[
        predicoes_xgboost["dia"].eq(dia)
    ].reset_index(drop=True)
    assert np.array_equal(referencia["bloco"].to_numpy(), blocos)
    assert np.array_equal(referencia["classe_real_nirrig"].to_numpy(), y_dia)

    previsto = np.full(len(y_dia), -1, dtype=int)
    probabilidade = np.full(len(y_dia), np.nan)
    dobra_por_linha = np.full(len(y_dia), -1, dtype=int)
    for dobra, (treino, teste) in enumerate(
        GroupKFold(n_splits=4).split(X_dia, y_dia, groups=blocos), start=1
    ):
        assert np.array_equal(referencia.loc[teste, "dobra"].to_numpy(), np.full(len(teste), dobra))
        # Mesmo conjunto de cinco bandas usado pelo modelo de referência nesta dobra.
        _, bandas_externas, _, _ = selecionar_top5_rf(
            X_dia[treino], y_dia[treino], comprimentos_onda,
            SEMENTE_CLASSIFICACAO + 100 * numero_dia + dobra,
        )
        partes = preparar_validacao_interna(
            X_dia[treino], y_dia[treino], blocos[treino], comprimentos_onda,
            SEMENTE_CLASSIFICACAO + 10000 + 100 * numero_dia + 10 * dobra,
        )
        estudo = otimizar_parametros_xgboost(
            partes, SEMENTE_CLASSIFICACAO + 20000 + 100 * numero_dia + dobra,
        )
        modelo = novo_xgboost(
            y_dia[treino], SEMENTE_CLASSIFICACAO + 100 * numero_dia + dobra,
        ).set_params(**estudo.best_params)
        modelo.fit(X_dia[np.ix_(treino, bandas_externas)], y_dia[treino])
        previsto[teste] = modelo.predict(X_dia[np.ix_(teste, bandas_externas)]).astype(int)
        probabilidade[teste] = modelo.predict_proba(
            X_dia[np.ix_(teste, bandas_externas)]
        )[:, 1]
        dobra_por_linha[teste] = dobra
        linhas_parametros_dobras.append({
            "dia": dia, "dobra": dobra,
            "bloco_teste": ", ".join(sorted(set(blocos[teste]))),
            "melhor_acuracia_balanceada_interna": estudo.best_value,
            "top5_treino_nm": ", ".join(map(str, comprimentos_onda[bandas_externas].astype(int))),
            **estudo.best_params,
        })

    assert (previsto >= 0).all() and np.isfinite(probabilidade).all()
    linhas_metricas_optuna.append({
        "dia": dia, "amostras": len(y_dia),
        **calcular_metricas(y_dia, previsto, probabilidade),
    })
    predicoes_optuna_lista.append(pd.DataFrame({
        "dia": dia, "bloco": blocos, "genotipo": meta_dia["genotipo"],
        "condicao_real": meta_dia["condicao"], "classe_real_nirrig": y_dia,
        "classe_predita_nirrig": previsto, "probabilidade_nirrig": probabilidade,
        "dobra": dobra_por_linha,
    }))

    # Busca final para o modelo que ficará disponível após a avaliação externa.
    partes_finais = preparar_validacao_interna(
        X_dia, y_dia, blocos, comprimentos_onda,
        SEMENTE_CLASSIFICACAO + 30000 + 100 * numero_dia,
    )
    estudo_final = otimizar_parametros_xgboost(
        partes_finais, SEMENTE_CLASSIFICACAO + 40000 + numero_dia,
    )
    estudo_final.trials_dataframe().to_csv(
        pasta_optuna / f"trials_{dia}.csv", sep=";", index=False,
    )
    linhas_parametros_finais.append({
        "dia": dia, "melhor_acuracia_balanceada_interna": estudo_final.best_value,
        "top5_nm": ", ".join(map(str, top5_rf_por_dia.loc[
            top5_rf_por_dia["dia"].eq(dia), "banda_nm"
        ])),
        **estudo_final.best_params,
    })
    bandas_finais = top5_rf_por_dia.loc[
        top5_rf_por_dia["dia"].eq(dia), "banda_nm"
    ].to_numpy(dtype=int)
    indices_finais = np.searchsorted(comprimentos_onda.astype(int), bandas_finais)
    assert np.array_equal(comprimentos_onda[indices_finais].astype(int), bandas_finais)
    modelo_final = novo_xgboost(
        y_dia, SEMENTE_CLASSIFICACAO + numero_dia,
    ).set_params(**estudo_final.best_params)
    modelo_final.fit(X_dia[:, indices_finais], y_dia)
    modelo_final.save_model(pasta_optuna / f"xgboost_{dia}.json")
    modelos_xgboost_optuna_por_dia[dia] = modelo_final
    print(f"{dia}: acurácia balanceada externa = "
          f"{linhas_metricas_optuna[-1]['acuracia_balanceada']:.3f}")

metricas_xgboost_optuna_por_dia = pd.DataFrame(linhas_metricas_optuna)
predicoes_xgboost_optuna = pd.concat(predicoes_optuna_lista, ignore_index=True)
parametros_optuna_por_dobra = pd.DataFrame(linhas_parametros_dobras)
parametros_optuna_finais = pd.DataFrame(linhas_parametros_finais)

metricas_xgboost_optuna_por_dia.to_csv(
    pasta_optuna / "metricas_xgboost_optuna_por_dia.csv", sep=";", index=False,
)
predicoes_xgboost_optuna.to_csv(
    pasta_optuna / "predicoes_xgboost_optuna.csv", sep=";", index=False,
)
parametros_optuna_por_dobra.to_csv(
    pasta_optuna / "parametros_por_dobra.csv", sep=";", index=False,
)
parametros_optuna_finais.to_csv(
    pasta_optuna / "parametros_finais_por_dia.csv", sep=";", index=False,
)


In [ ]:
comparacao_optuna = metricas_xgboost_por_dia[[
    "dia", "amostras", "acuracia", "acuracia_balanceada", "f1_nirrig", "auc_roc"
]].merge(
    metricas_xgboost_optuna_por_dia[[
        "dia", "acuracia", "acuracia_balanceada", "f1_nirrig", "auc_roc"
    ]], on="dia", suffixes=("_base", "_optuna"), validate="one_to_one",
)
for metrica in ("acuracia", "acuracia_balanceada", "f1_nirrig", "auc_roc"):
    comparacao_optuna[f"delta_{metrica}"] = (
        comparacao_optuna[f"{metrica}_optuna"] - comparacao_optuna[f"{metrica}_base"]
    )
comparacao_optuna.to_csv(
    pasta_optuna / "comparacao_baseline_optuna.csv", sep=";", index=False,
)
display(comparacao_optuna.round(3))
delta = comparacao_optuna["delta_acuracia_balanceada"]
print(
    f"Optuna melhorou {int((delta > 1e-12).sum())} dias, "
    f"empatou {int((delta.abs() <= 1e-12).sum())} e "
    f"piorou {int((delta < -1e-12).sum())}; "
    f"variação média da acurácia balanceada = {delta.mean():+.3f}."
)
display(parametros_optuna_finais.round(3))

fig, eixo = plt.subplots(figsize=(11, 5))
x = np.arange(len(comparacao_optuna))
largura = 0.38
eixo.bar(
    x - largura / 2, comparacao_optuna["acuracia_balanceada_base"],
    width=largura, label="XGBoost base", color="#2a78d6",
)
eixo.bar(
    x + largura / 2, comparacao_optuna["acuracia_balanceada_optuna"],
    width=largura, label="XGBoost + Optuna", color="#eb6834",
)
eixo.set_xticks(x, comparacao_optuna["dia"])
eixo.set_ylim(0, 1.05)
eixo.set_xlabel("Dia")
eixo.set_ylabel("Acurácia balanceada fora do treino")
eixo.set_title("Classificação por dia: modelo base × Optuna", fontweight="bold")
eixo.grid(axis="y", alpha=0.25)
eixo.legend(frameon=False)
fig.tight_layout()
fig.savefig(pasta_optuna / "comparacao_acuracia_balanceada.png", dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)


In [ ]:
matrizes_optuna_por_dia = {}
estatisticas_optuna_linhas = []
for dia in sorted(predicoes_xgboost_optuna["dia"].unique()):
    dados = predicoes_xgboost_optuna.loc[predicoes_xgboost_optuna["dia"].eq(dia)]
    real = dados["classe_real_nirrig"].to_numpy(dtype=int)
    previsto = dados["classe_predita_nirrig"].to_numpy(dtype=int)
    matriz = confusion_matrix(real, previsto, labels=[0, 1])
    linha_metricas = metricas_xgboost_optuna_por_dia.set_index("dia").loc[dia]
    assert matriz.tolist() == [
        [int(linha_metricas["tn"]), int(linha_metricas["fp"])],
        [int(linha_metricas["fn"]), int(linha_metricas["tp"])],
    ]
    matrizes_optuna_por_dia[dia] = matriz
    pd.DataFrame(matriz, index=classes, columns=classes).to_csv(
        pasta_optuna / f"matriz_confusao_{dia}.csv", sep=";",
    )
    relatorio = classification_report(
        real, previsto, labels=[0, 1], target_names=list(classes),
        output_dict=True, zero_division=0,
    )
    for classe in (*classes, "macro avg", "weighted avg"):
        valores = relatorio[classe]
        estatisticas_optuna_linhas.append({
            "dia": dia, "classe": classe,
            "precisao": valores["precision"],
            "sensibilidade_recall": valores["recall"],
            "f1": valores["f1-score"],
            "suporte": int(valores["support"]),
        })
estatisticas_optuna_por_classe = pd.DataFrame(estatisticas_optuna_linhas)
estatisticas_optuna_por_classe.to_csv(
    pasta_optuna / "estatisticas_por_classe_dia.csv", sep=";", index=False,
)

fig, eixos = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
maximo = max(int(matriz.max()) for matriz in matrizes_optuna_por_dia.values())
for eixo, (dia, matriz) in zip(eixos.flat, matrizes_optuna_por_dia.items()):
    eixo.imshow(matriz, cmap="Blues", vmin=0, vmax=maximo)
    eixo.set_title(f"{dia}  ·  n={matriz.sum()}", fontweight="bold")
    eixo.set_xticks([0, 1], classes)
    eixo.set_yticks([0, 1], classes)
    eixo.set_xlabel("Predita")
    eixo.set_ylabel("Real")
    for linha in range(2):
        for coluna in range(2):
            n = int(matriz[linha, coluna])
            percentual = 100 * n / matriz[linha].sum()
            eixo.text(
                coluna, linha, f"{n}\n({percentual:.1f}%)",
                ha="center", va="center", fontsize=11,
                color="white" if n > maximo / 2 else "#1d2a35",
            )
for eixo in list(eixos.flat)[len(matrizes_optuna_por_dia):]:
    eixo.axis("off")
fig.savefig(pasta_optuna / "matrizes_confusao_optuna_por_dia.png", dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)


## Parte 4 — predição diária de parâmetros fisiológicos

Os alvos são **clorofila total (CHL total), clorofila A, clorofila B, condutância estomática (gs), conteúdo relativo de água (CRA) e taxa de transporte de elétrons (ETR)**. A fonte fisiológica é `ParametrosFisiologicos.xlsx`, com 24 repetições por data quando todas as medidas estão disponíveis (3 genótipos × 2 condições × 4 blocos). A planilha não contém uma coluna de bloco: dentro de cada combinação genótipo × condição, a ordem das quatro linhas é mapeada a B1–B4, como nos scripts do projeto. O pareamento com o espectro é feito pela chave **bloco × genótipo × condição** e pela data correspondente. Antes do pareamento, as leituras espectrais da manhã de cada parcela são reduzidas à média por banda; assim, a unidade do PLSR é a parcela, e não cada varredura.

| Parâmetro | Coletas fisiológicas | Espectro e seleção |
|---|---|---|
| CHL total, clorofila A, clorofila B, gs | 23–27/02, 02/03 e 03/03 | Espectro corrigido, 400–2450 nm, Savitzky–Golay e SNV; Spearman bilateral com o alvo, `p < 0,001`, nas datas reunidas; até cinco bandas globais e PLSR independente por data. |
| CRA | 24/02, 02/03 e 03/03 | Mesmo pré-processamento e filtro `p < 0,001` nas datas disponíveis; até cinco bandas globais e PLSR diário. A observação sem CRA em 02/03 é excluída. |
| ETR | 23/02, 02/03 e 03/03 | Reflectância original de 350–2500 nm; Spearman bilateral **por data** com `p < 0,001`, representantes em intervalos de 10 nm e retenção de VIP ≥ 1. |

**CHL total, A, B, gs e CRA.** Para cada parâmetro, o Spearman com o alvo é calculado em todas as suas parcelas disponíveis, reunindo as datas daquele parâmetro. Somente bandas com `p < 0,001` prosseguem. As bandas vizinhas aprovadas são agrupadas por correlação de Spearman `|ρ| > 0,80`, com ligação completa e extensão menor que 10 nm. Em cada grupo, fica a banda de maior `|ρ|` com o parâmetro. Um PLSR escalonado reúne as datas disponíveis; o número de componentes (até cinco) minimiza o RMSE em validação por bloco. As representantes são ordenadas pelo módulo do coeficiente PLSR padronizado. Percorrendo o ranking, mantêm-se **até cinco bandas** com `|r| < 0,80` de Pearson entre si. Essas bandas alimentam um novo PLSR para **cada data**, com componentes escolhidos pelo menor RMSE ao deixar um bloco de fora. Se a restrição de redundância deixar menos de cinco bandas, o modelo usa as restantes; isso ocorre para clorofila A nesta base.

**ETR.** Em cada data, o Spearman bilateral relaciona cada banda bruta ao ETR. Somente bandas com `p < 0,001` entram em intervalos fixos de 10 nm; a maior `|ρ|` representa cada intervalo. Um PLSR escalonado escolhe até seis componentes pelo RMSE da validação por bloco. As representantes com VIP ≥ 1 formam o modelo final daquele dia. Se nenhuma banda atingir o limiar, não se ajusta PLSR para a data.

As tabelas mostram `R²` e RMSE de ajuste e de validação por bloco. **Os valores de validação são diagnósticos**: a seleção de bandas e a escolha de componentes usam os dados antes da predição nas dobras; portanto, os valores podem ser otimistas e não representam validação externa aninhada. O filtro de `p < 0,001` também é aplicado antes das dobras. As seções finais mostram os valores observados e preditos por dia e parâmetro.


### 4.1. Planilha fisiológica e pareamento

A célula seguinte lê `ParametrosFisiologicos.xlsx`, normaliza os nomes das colunas, mapeia as quatro repetições de cada genótipo × condição para B1–B4 e calcula a média das leituras matinais de cada parcela em cada dia. O pareamento exige uma linha espectral por parcela; uma medida fisiológica ausente é excluída somente daquele alvo e daquela data.


In [ ]:
from scipy.stats import rankdata, spearmanr
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold

MAPA_FISIOLOGICO = {
    "23/02": "D02M", "24/02": "D03M", "25/02": "D04M",
    "26/02": "D05M", "27/02": "D06M", "02/03": "D09M", "03/03": "D10M",
}
DATAS_ETR = ("23/02", "02/03", "03/03")
DATAS_CRA = ("24/02", "02/03", "03/03")
ALVOS_FISIOLOGICOS = {
    "CHL total": ("CHL TOTAL", tuple(MAPA_FISIOLOGICO)),
    "Clorofila A": ("CHL A", tuple(MAPA_FISIOLOGICO)),
    "Clorofila B": ("CHL B", tuple(MAPA_FISIOLOGICO)),
    "gs": ("gs", tuple(MAPA_FISIOLOGICO)),
    "CRA": ("CRA", DATAS_CRA),
    "ETR": ("ETR", DATAS_ETR),
}
PASTA_FISIOLOGIA = ROOT / "script_final/outputs/fisiologia"
PASTA_FISIOLOGIA.mkdir(parents=True, exist_ok=True)

fisiologia = pd.read_excel(ROOT / "ParametrosFisiologicos.xlsx")
fisiologia.columns = [" ".join(str(c).replace(".", "/").split()) for c in fisiologia.columns]
fisiologia["genotipo"] = fisiologia["Genótipo"]
fisiologia["condicao"] = fisiologia["Condição"].replace({
    "IRR": "IRRIG", "NIR": "NIRRIG", "NIRR": "NIRRIG",
})
fisiologia["bloco"] = (
    fisiologia.groupby(["genotipo", "condicao"], sort=False).cumcount()
    .map(lambda i: f"B{i + 1}")
)
assert len(fisiologia) == 24
assert fisiologia.groupby(["genotipo", "condicao"]).size().eq(4).all()
CHAVES_FISIOLOGIA = ["bloco", "genotipo", "condicao"]

bandas_fisiologia_norm = comprimentos_onda.astype(int)
colunas_fisiologia_norm = [str(b) for b in bandas_fisiologia_norm]
bandas_fisiologia_bruto = np.array(sorted(
    int(c) for c in base_limpa.columns if c.isdigit()
))
colunas_fisiologia_bruto = [str(b) for b in bandas_fisiologia_bruto]

def medias_espectrais_por_parcela(base: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    usadas = ["data_coleta", *CHAVES_FISIOLOGIA, *colunas]
    manha = base.loc[base["turno"].eq("manha"), usadas].copy()
    return manha.groupby(["data_coleta", *CHAVES_FISIOLOGIA], as_index=False)[colunas].mean()


espectros_fisiologia_norm = medias_espectrais_por_parcela(
    base_normalizada, colunas_fisiologia_norm
)
espectros_fisiologia_bruto = medias_espectrais_por_parcela(
    base_limpa, colunas_fisiologia_bruto
)


def parear_fisiologia(alvo: str, data: str, espectros: pd.DataFrame) -> pd.DataFrame:
    prefixo, _ = ALVOS_FISIOLOGICOS[alvo]
    coluna = f"{prefixo} ({data})"
    if coluna not in fisiologia:
        raise KeyError(f"Coluna fisiológica ausente: {coluna}")
    medida = fisiologia[CHAVES_FISIOLOGIA].copy()
    medida["observado"] = pd.to_numeric(
        fisiologia[coluna].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    coleta = MAPA_FISIOLOGICO[data]
    espectro_dia = espectros.loc[espectros["data_coleta"].eq(coleta)].drop(
        columns="data_coleta"
    )
    dados = medida.merge(
        espectro_dia, on=CHAVES_FISIOLOGIA, how="inner", validate="one_to_one"
    )
    if len(dados) != 24:
        raise ValueError(f"{alvo} {data}: pareamento incompleto ({len(dados)}/24).")
    return dados.dropna(subset=["observado"]).reset_index(drop=True)


contagens_fisiologia = pd.DataFrame([
    {"parametro": alvo, "data": data, "coleta_espectral": MAPA_FISIOLOGICO[data],
     "pares_validos": len(parear_fisiologia(
         alvo, data, espectros_fisiologia_bruto if alvo == "ETR" else espectros_fisiologia_norm
     ))}
    for alvo, (_, datas) in ALVOS_FISIOLOGICOS.items() for data in datas
])
display(contagens_fisiologia)


### 4.2. Filtros e funções PLSR

As funções abaixo implementam o agrupamento Spearman com `|ρ| > 0,80` e janela menor que 10 nm, a escolha de componentes pelo menor RMSE em `GroupKFold` por bloco, o cálculo de VIP e a predição deixando um bloco de fora. Elas usam os dados já preparados neste notebook.


In [ ]:
def grupos_spearman_fisiologia(X: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Ligação completa local: |ρ| > 0,80 em grupos com extensão <10 nm."""
    postos = rankdata(X, axis=0, method="average")
    z = postos - postos.mean(axis=0)
    norma = np.sqrt((z ** 2).sum(axis=0))
    norma[norma == 0] = np.inf
    z /= norma
    grupos = np.zeros(len(w), dtype=int)
    membros = [0]
    atual = 0
    for j in range(1, len(w)):
        correlacionada = (
            w[j] - w[membros[0]] < 10
            and np.all(np.abs(z[:, membros].T @ z[:, j]) > 0.80)
        )
        if correlacionada:
            membros.append(j)
        else:
            atual += 1
            membros = [j]
        grupos[j] = atual
    return grupos


def componentes_plsr_por_bloco(
    X: np.ndarray, y: np.ndarray, blocos: np.ndarray, max_componentes: int,
) -> tuple[int, pd.DataFrame]:
    dobras = list(GroupKFold(n_splits=len(np.unique(blocos))).split(X, y, blocos))
    limite = min(max_componentes, X.shape[1], min(len(treino) - 1 for treino, _ in dobras))
    linhas = []
    for n in range(1, limite + 1):
        previsto = np.full(len(y), np.nan)
        for treino, teste in dobras:
            modelo = PLSRegression(n_components=n, scale=True).fit(X[treino], y[treino])
            previsto[teste] = modelo.predict(X[teste]).ravel()
        linhas.append({
            "componentes": n, "r2_cv_bloco": r2_score(y, previsto),
            "rmse_cv_bloco": mean_squared_error(y, previsto) ** 0.5,
        })
    tabela = pd.DataFrame(linhas)
    return int(tabela.loc[tabela["rmse_cv_bloco"].idxmin(), "componentes"]), tabela


def prever_plsr_por_bloco(X: np.ndarray, y: np.ndarray, blocos: np.ndarray, n: int) -> np.ndarray:
    previsto = np.full(len(y), np.nan)
    for treino, teste in GroupKFold(n_splits=len(np.unique(blocos))).split(X, y, blocos):
        previsto[teste] = PLSRegression(n_components=n, scale=True).fit(
            X[treino], y[treino]
        ).predict(X[teste]).ravel()
    return previsto


def correlacao_spearman_alvo(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    postos_x = rankdata(X, axis=0, method="average")
    postos_y = rankdata(y, method="average")
    zx = postos_x - postos_x.mean(axis=0)
    zy = postos_y - postos_y.mean()
    norma_x = np.sqrt((zx ** 2).sum(axis=0))
    norma_x[norma_x == 0] = np.inf
    norma_y = np.sqrt((zy ** 2).sum())
    return zx.T @ zy / (norma_x * norma_y)


def vip_plsr(modelo: PLSRegression) -> np.ndarray:
    t, pesos, q = modelo.x_scores_, modelo.x_weights_, modelo.y_loadings_
    ssy = (q ** 2).ravel() * (t ** 2).sum(axis=0)
    pesos_norm = pesos / np.linalg.norm(pesos, axis=0, keepdims=True)
    return np.sqrt(pesos.shape[0] * ((pesos_norm ** 2) * ssy).sum(axis=1) / ssy.sum())


def resumir_plsr(alvo: str, data: str, coleta: str, X: np.ndarray, y: np.ndarray,
                 blocos: np.ndarray, bandas: np.ndarray, n: int,
                 representantes: int, metodo: str) -> tuple[dict, np.ndarray, PLSRegression]:
    modelo = PLSRegression(n_components=n, scale=True).fit(X, y)
    ajuste = modelo.predict(X).ravel()
    previsto = prever_plsr_por_bloco(X, y, blocos, n)
    resumo = {
        "parametro": alvo, "data": data, "coleta_espectral": coleta,
        "metodo": metodo, "n": len(y), "bandas_entrada": len(bandas),
        "representantes": representantes, "bandas_modelo": X.shape[1],
        "componentes": n, "r2_ajuste": r2_score(y, ajuste),
        "rmse_ajuste": mean_squared_error(y, ajuste) ** 0.5,
        "r2_cv_bloco": r2_score(y, previsto),
        "rmse_cv_bloco": mean_squared_error(y, previsto) ** 0.5,
        "status": "Modelo ajustado",
    }
    return resumo, previsto, modelo


### 4.3. Filtro p < 0,001 e seleção global para CHL total, clorofilas A/B, gs e CRA

Para cada parâmetro, o código reúne apenas suas datas disponíveis e testa a correlação de Spearman bilateral de cada banda com o alvo. Somente bandas com `p < 0,001` entram no agrupamento local `|ρ| > 0,80`. Em cada grupo fica a banda com maior `|ρ|` com o alvo. O PLSR global ordena as representantes pelo módulo do coeficiente padronizado; até cinco bandas são mantidas quando têm `|r de Pearson| < 0,80` com todas as já escolhidas. Depois, o PLSR de **cada data** usa essas bandas e escolhe seu próprio número de componentes.

O arquivo `spearman_global.csv` de cada parâmetro guarda ρ, p e a decisão do filtro por banda; `selecao_bandas_globais.csv` resume quantas bandas passaram e quantas entraram no modelo.


In [ ]:
modelos_fisiologia_diarios = {}
resumos_fisiologia, predicoes_fisiologia = [], []
top5_fisiologia_global = {}
selecao_bandas_fisiologia = []

def executar_plsr_global(alvo: str) -> None:
    _, datas = ALVOS_FISIOLOGICOS[alvo]
    partes = []
    for data in datas:
        parte = parear_fisiologia(alvo, data, espectros_fisiologia_norm)
        parte.insert(0, "data", data)
        partes.append(parte)
    dados_globais = pd.concat(partes, ignore_index=True)
    X = dados_globais[colunas_fisiologia_norm].to_numpy(dtype=float)
    y = dados_globais["observado"].to_numpy(dtype=float)
    blocos = dados_globais["bloco"].to_numpy()

    rho_y = np.empty(X.shape[1], dtype=float)
    p_valor = np.empty(X.shape[1], dtype=float)
    for j in range(X.shape[1]):
        rho_y[j], p_valor[j] = spearmanr(X[:, j], y)
    validas = np.isfinite(rho_y) & np.isfinite(p_valor) & (p_valor < 0.001)
    pasta_alvo = PASTA_FISIOLOGIA / alvo.lower().replace(" ", "_")
    pasta_alvo.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "banda_nm": bandas_fisiologia_norm,
        "rho_spearman_alvo": rho_y, "p_spearman_alvo": p_valor,
        "passa_p_0001": validas,
    }).to_csv(pasta_alvo / "spearman_global.csv", sep=";", index=False)
    if not validas.any():
        raise ValueError(f"{alvo}: nenhuma banda passou p < 0,001.")

    X_validas = X[:, validas]
    bandas_validas = bandas_fisiologia_norm[validas]
    rho_validas = rho_y[validas]
    p_validas = p_valor[validas]
    grupos_rho = grupos_spearman_fisiologia(X_validas, bandas_validas)
    representantes = np.array([
        indices[np.nanargmax(np.abs(rho_validas[indices]))]
        for grupo in np.unique(grupos_rho)
        if (indices := np.flatnonzero(grupos_rho == grupo)).size
    ], dtype=int)
    Xr = X_validas[:, representantes]
    n_ranking, _ = componentes_plsr_por_bloco(Xr, y, blocos, max_componentes=5)
    modelo_ranking = PLSRegression(n_components=n_ranking, scale=True).fit(Xr, y)
    coeficiente = np.asarray(modelo_ranking.coef_).reshape(-1)
    beta = coeficiente * modelo_ranking._x_std / modelo_ranking._y_std[0]
    ordem = np.argsort(np.abs(beta))[::-1]
    corr_pearson = np.corrcoef(Xr, rowvar=False)
    escolhidas = []
    for posicao in ordem:
        if not escolhidas or np.all(np.abs(corr_pearson[posicao, escolhidas]) < 0.80):
            escolhidas.append(int(posicao))
        if len(escolhidas) == 5:
            break
    if not escolhidas:
        raise ValueError(f"{alvo}: não foi possível selecionar bandas.")
    bandas_top5 = bandas_validas[representantes[escolhidas]]
    top5_fisiologia_global[alvo] = bandas_top5
    selecao_bandas_fisiologia.append({
        "parametro": alvo, "amostras": len(y),
        "bandas_testadas": len(bandas_fisiologia_norm),
        "bandas_p_menor_0001": int(validas.sum()),
        "representantes": len(representantes),
        "bandas_selecionadas": len(bandas_top5),
        "bandas_nm": ", ".join(map(str, bandas_top5)),
    })
    pd.DataFrame({
        "parametro": alvo, "posicao": np.arange(1, len(escolhidas) + 1),
        "banda_nm": bandas_top5,
        "coeficiente_padronizado": beta[escolhidas],
        "rho_spearman_alvo": rho_validas[representantes[escolhidas]],
        "p_spearman_alvo": p_validas[representantes[escolhidas]],
    }).to_csv(pasta_alvo / "top5_bandas_globais.csv", sep=";", index=False)

    for data in datas:
        dia = dados_globais.loc[dados_globais["data"].eq(data)].reset_index(drop=True)
        X_dia = dia[[str(b) for b in bandas_top5]].to_numpy(dtype=float)
        y_dia = dia["observado"].to_numpy(dtype=float)
        blocos_dia = dia["bloco"].to_numpy()
        n, componentes = componentes_plsr_por_bloco(
            X_dia, y_dia, blocos_dia, max_componentes=5
        )
        resumo, previsto, modelo = resumir_plsr(
            alvo, data, MAPA_FISIOLOGICO[data], X_dia, y_dia, blocos_dia,
            bandas_validas, n, len(representantes),
            "Spearman p < 0,001 / até 5 bandas globais / PLSR diário",
        )
        resumos_fisiologia.append(resumo)
        modelos_fisiologia_diarios[(alvo, data)] = modelo
        predicoes_fisiologia.append(dia[[*CHAVES_FISIOLOGIA, "data", "observado"]].assign(
            parametro=alvo, predito_cv_bloco=previsto, residuo_cv_bloco=y_dia - previsto,
        ))
        componentes.to_csv(
            pasta_alvo / f"componentes_{data.replace('/', '_')}.csv", sep=";", index=False
        )
    print(f"{alvo}: {validas.sum()} bandas com p < 0,001; "
          f"{len(representantes)} representantes; "
          f"{len(bandas_top5)} bandas selecionadas = {', '.join(map(str, bandas_top5))}")


def mostrar_resumo_fisiologico(alvo: str) -> None:
    linhas = [r for r in resumos_fisiologia if r["parametro"] == alvo]
    display(pd.DataFrame(linhas)[[
        "parametro", "data", "n", "bandas_modelo", "componentes",
        "r2_cv_bloco", "rmse_cv_bloco", "status",
    ]].round(3))


### 4.4. CHL total

**Protocolo:** sete coletas (23–27/02, 02/03 e 03/03); reflectância corrigida e normalizada por SNV. **Filtro:** Spearman bilateral com CHL total nas datas reunidas, `p < 0,001`, seguido do agrupamento local `|ρ| > 0,80`. **Seleção:** até cinco bandas do ranking PLSR global, com `|r de Pearson| < 0,80` entre as escolhidas. **Ajuste:** um PLSR escalonado por data, com até cinco componentes escolhidos pelo menor RMSE em CV por bloco.


In [ ]:
executar_plsr_global("CHL total")
mostrar_resumo_fisiologico("CHL total")


### 4.5. Clorofila A

**Protocolo:** as mesmas sete coletas e o mesmo espectro SNV. **Filtro:** Spearman bilateral com clorofila A nas datas reunidas, `p < 0,001`, seguido de agrupamento local `|ρ| > 0,80` em menos de 10 nm. **Seleção:** até cinco bandas globais pelo coeficiente PLSR padronizado, evitando pares com `|r de Pearson| ≥ 0,80`; nesta base, quatro bandas atendem aos critérios. **Ajuste:** PLSR independente por dia, com componentes escolhidos por CV deixando um bloco de fora.


In [ ]:
executar_plsr_global("Clorofila A")
mostrar_resumo_fisiologico("Clorofila A")


### 4.6. Clorofila B

**Protocolo:** sete coletas e espectro SNV. **Filtro:** Spearman bilateral com clorofila B nas datas reunidas, `p < 0,001`, seguido de agrupamento local `|ρ| > 0,80` com ligação completa. **Seleção:** até cinco bandas do PLSR global, mantendo correlação de Pearson abaixo de 0,80 em módulo entre elas. **Ajuste:** PLSR por data, escalonado e validado por exclusão de bloco.


In [ ]:
executar_plsr_global("Clorofila B")
mostrar_resumo_fisiologico("Clorofila B")


### 4.7. gs — condutância estomática

**Protocolo:** sete coletas; média espectral por parcela após SNV. **Filtro:** Spearman bilateral com gs nas datas reunidas, `p < 0,001`, seguido do agrupamento local `|ρ| > 0,80` entre bandas vizinhas. **Seleção:** até cinco bandas globais ranqueadas pelo coeficiente PLSR padronizado, removendo redundância por Pearson `|r| ≥ 0,80`. **Ajuste:** PLSR separado em cada data, com componentes escolhidos pelo RMSE em CV por bloco.


In [ ]:
executar_plsr_global("gs")
mostrar_resumo_fisiologico("gs")


### 4.8. CRA — conteúdo relativo de água

**Protocolo:** somente 24/02, 02/03 e 03/03; o valor ausente de CRA em 02/03 é descartado após o pareamento, ficando 23 parcelas nessa data. **Filtro:** Spearman bilateral com CRA nas datas disponíveis reunidas, `p < 0,001`, seguido de agrupamento local `|ρ| > 0,80`. **Seleção:** até cinco bandas globais pelo coeficiente PLSR padronizado e restrição de Pearson `|r| < 0,80`. **Ajuste:** PLSR diário escalonado, com componentes por RMSE em validação por bloco.


In [ ]:
executar_plsr_global("CRA")
mostrar_resumo_fisiologico("CRA")


### 4.9. ETR — taxa de transporte de elétrons

**Protocolo:** 23/02, 02/03 e 03/03, usando a reflectância original de 350–2500 nm. **Filtro:** correlação de Spearman entre cada banda e ETR, com `p < 0,001`. **Seleção:** uma representante por intervalo fixo de 10 nm, escolhida pelo maior `|ρ|`; após um PLSR inicial, ficam apenas bandas com VIP ≥ 1. **Ajuste:** PLSR escalonado por data, até seis componentes pelo menor RMSE em CV por bloco. Uma data sem bandas aprovadas não recebe modelo.


In [ ]:
# ETR conserva a reflectância bruta e o filtro p < 0,001 do protocolo específico.
def executar_etr(data: str) -> None:
    dados = parear_fisiologia("ETR", data, espectros_fisiologia_bruto)
    X = dados[colunas_fisiologia_bruto].to_numpy(dtype=float)
    y = dados["observado"].to_numpy(dtype=float)
    blocos = dados["bloco"].to_numpy()
    rho = np.empty(X.shape[1])
    p_valor = np.empty(X.shape[1])
    for j in range(X.shape[1]):
        rho[j], p_valor[j] = spearmanr(X[:, j], y)
    validas = np.isfinite(rho) & np.isfinite(p_valor) & (p_valor < 0.001)
    intervalos = (bandas_fisiologia_bruto - bandas_fisiologia_bruto.min()) // 10
    representantes = np.array([
        indices[np.argmax(np.abs(rho[indices]))]
        for intervalo in np.unique(intervalos[validas])
        if (indices := np.flatnonzero((intervalos == intervalo) & validas)).size
    ], dtype=int)
    pasta_etr = PASTA_FISIOLOGIA / "etr"
    pasta_etr.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "banda_nm": bandas_fisiologia_bruto, "rho_spearman_etr": rho,
        "p_spearman_etr": p_valor, "passa_p_0001": validas,
    }).to_csv(pasta_etr / f"spearman_{data.replace('/', '_')}.csv", sep=";", index=False)
    if not len(representantes):
        resumos_fisiologia.append({
            "parametro": "ETR", "data": data,
            "coleta_espectral": MAPA_FISIOLOGICO[data],
            "metodo": "Spearman p < 0,001 / iPLS / VIP / PLSR diário",
            "n": len(y), "bandas_entrada": len(bandas_fisiologia_bruto),
            "representantes": 0, "bandas_modelo": 0,
            "componentes": np.nan, "r2_ajuste": np.nan, "rmse_ajuste": np.nan,
            "r2_cv_bloco": np.nan, "rmse_cv_bloco": np.nan,
            "status": "Sem bandas com p < 0,001",
        })
        print(f"ETR {data}: nenhuma banda passou p < 0,001; modelo não ajustado.")
        return

    Xr = X[:, representantes]
    n_inicial, _ = componentes_plsr_por_bloco(Xr, y, blocos, max_componentes=6)
    vip_inicial = vip_plsr(
        PLSRegression(n_components=n_inicial, scale=True).fit(Xr, y)
    )
    manter = vip_inicial >= 1
    if not manter.any():
        manter[np.argmax(vip_inicial)] = True
    bandas_finais = bandas_fisiologia_bruto[representantes[manter]]
    Xfinal = Xr[:, manter]
    n_final, componentes = componentes_plsr_por_bloco(
        Xfinal, y, blocos, max_componentes=6
    )
    resumo, previsto, modelo = resumir_plsr(
        "ETR", data, MAPA_FISIOLOGICO[data], Xfinal, y, blocos,
        bandas_fisiologia_bruto, n_final, len(representantes),
        "Spearman p < 0,001 / iPLS / VIP / PLSR diário",
    )
    resumos_fisiologia.append(resumo)
    modelos_fisiologia_diarios[("ETR", data)] = modelo
    predicoes_fisiologia.append(dados[[*CHAVES_FISIOLOGIA, "observado"]].assign(
        data=data, parametro="ETR", predito_cv_bloco=previsto,
        residuo_cv_bloco=y - previsto,
    ))
    pd.DataFrame({
        "banda_nm": bandas_finais, "vip_selecao": vip_inicial[manter],
        "vip_modelo_final": vip_plsr(modelo),
        "rho_spearman_etr": rho[representantes[manter]],
    }).sort_values("vip_modelo_final", ascending=False).to_csv(
        pasta_etr / f"bandas_vip_{data.replace('/', '_')}.csv", sep=";", index=False
    )
    componentes.to_csv(
        pasta_etr / f"componentes_{data.replace('/', '_')}.csv", sep=";", index=False
    )
    print(f"ETR {data}: {len(representantes)} intervalos; {len(bandas_finais)} bandas VIP ≥ 1.")


In [ ]:
for data in DATAS_ETR:
    executar_etr(data)
mostrar_resumo_fisiologico("ETR")


### 4.10. Resultados diários

O resumo reúne R² e RMSE de ajuste e de CV por bloco; o arquivo de predições guarda observado, predito e resíduo por parcela. As métricas de CV continuam **diagnósticas**, pois o filtro e a seleção de bandas são feitos antes das dobras.


In [ ]:
resumo_fisiologia_diaria = pd.DataFrame(resumos_fisiologia)
predicoes_fisiologia_diaria = pd.concat(predicoes_fisiologia, ignore_index=True)
resumo_fisiologia_diaria.to_csv(
    PASTA_FISIOLOGIA / "resumo_predicao_diaria.csv", sep=";", index=False
)
predicoes_fisiologia_diaria.to_csv(
    PASTA_FISIOLOGIA / "predicoes_cv_bloco.csv", sep=";", index=False
)
display(resumo_fisiologia_diaria[[
    "parametro", "data", "n", "bandas_modelo", "componentes",
    "r2_cv_bloco", "rmse_cv_bloco", "status",
]].round(3))

fig, eixos = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for eixo, alvo in zip(eixos.flat, ALVOS_FISIOLOGICOS):
    dados = resumo_fisiologia_diaria.loc[
        resumo_fisiologia_diaria["parametro"].eq(alvo)
    ]
    eixo.plot(dados["data"], dados["r2_cv_bloco"], marker="o", color="#2a78d6")
    eixo.axhline(0, color="#555555", linewidth=0.8)
    eixo.set_title(alvo, fontweight="bold")
    eixo.set_ylabel("R² CV por bloco (diagnóstico)")
    eixo.tick_params(axis="x", rotation=45)
    eixo.grid(alpha=0.25)
fig.savefig(PASTA_FISIOLOGIA / "r2_cv_por_parametro_e_dia.png", dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)

selecao_bandas_fisiologia = pd.DataFrame(selecao_bandas_fisiologia)
selecao_bandas_fisiologia.to_csv(
    PASTA_FISIOLOGIA / "selecao_bandas_globais.csv", sep=";", index=False
)
display(selecao_bandas_fisiologia)


### 4.11. Gráficos da predição por parâmetro e data

Cada painel compara o valor observado com a predição obtida ao deixar um bloco de fora. A linha diagonal representa predição perfeita; as cores distinguem parcelas irrigadas e não irrigadas. O título de cada painel traz R² e RMSE da CV por bloco. Datas sem bandas aprovadas aparecem explicitamente sem modelo. Uma figura por parâmetro é salva em `script_final/outputs/fisiologia/`.


In [ ]:
CORES_CONDICAO_FISIOLOGIA = {"IRRIG": "#2374ab", "NIRRIG": "#d95f02"}

def grafico_predicao_fisiologica(alvo: str) -> None:
    _, datas = ALVOS_FISIOLOGICOS[alvo]
    n_colunas = min(4, len(datas))
    n_linhas = int(np.ceil(len(datas) / n_colunas))
    fig, eixos = plt.subplots(
        n_linhas, n_colunas, figsize=(4.3 * n_colunas, 4 * n_linhas),
        squeeze=False, constrained_layout=True,
    )
    for eixo, data in zip(eixos.flat, datas):
        metrica = resumo_fisiologia_diaria.loc[
            resumo_fisiologia_diaria["parametro"].eq(alvo)
            & resumo_fisiologia_diaria["data"].eq(data)
        ].iloc[0]
        dia = predicoes_fisiologia_diaria.loc[
            predicoes_fisiologia_diaria["parametro"].eq(alvo)
            & predicoes_fisiologia_diaria["data"].eq(data)
        ]
        if dia.empty:
            eixo.text(0.5, 0.5, metrica["status"], ha="center", va="center",
                      transform=eixo.transAxes, color="#777777")
            eixo.set_title(data)
            eixo.set_xticks([])
            eixo.set_yticks([])
            continue
        valores = np.r_[dia["observado"].to_numpy(),
                        dia["predito_cv_bloco"].to_numpy()]
        amplitude = np.ptp(valores)
        margem = max(0.05 * amplitude, 0.01)
        minimo, maximo = np.min(valores) - margem, np.max(valores) + margem
        for condicao, cor in CORES_CONDICAO_FISIOLOGIA.items():
            parcela = dia.loc[dia["condicao"].eq(condicao)]
            if not parcela.empty:
                eixo.scatter(parcela["observado"], parcela["predito_cv_bloco"],
                             label=condicao, color=cor, alpha=0.78, s=32)
        eixo.plot([minimo, maximo], [minimo, maximo], "--",
                  color="#555555", linewidth=1)
        eixo.set_xlim(minimo, maximo)
        eixo.set_ylim(minimo, maximo)
        eixo.set_aspect("equal", adjustable="box")
        eixo.set_title(f"{data} | R²={metrica['r2_cv_bloco']:.2f}; "
                       f"RMSE={metrica['rmse_cv_bloco']:.2f}")
        eixo.set_xlabel("Observado")
        eixo.set_ylabel("Predito (CV por bloco)")
        eixo.grid(alpha=0.2)
    for eixo in list(eixos.flat)[len(datas):]:
        eixo.set_visible(False)
    handles, labels = [], []
    for eixo in eixos.flat:
        handles, labels = eixo.get_legend_handles_labels()
        if handles:
            break
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=2,
                   bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(f"{alvo}: observado × predito por dia", fontweight="bold")
    nome = alvo.lower().replace(" ", "_")
    fig.savefig(PASTA_FISIOLOGIA / f"predicao_{nome}_por_dia.png",
                dpi=170, bbox_inches="tight")
    display(fig)
    plt.close(fig)

for alvo in ALVOS_FISIOLOGICOS:
    grafico_predicao_fisiologica(alvo)
